# Aim 1 Repeated Measures -- QC + Autopay

## Run QC


In [1]:
### Add info here
downloads_path = "/Users/msg74/Downloads"
expensesheetpath = "/Users/msg74/Desktop/powers_lab/Analysis/Aim1_rpt/python_scripts/qc/expensesheets"
myname = "Maximillian S. Greenwald"

#Import libraries
import pandas as pd
import numpy as np
import math
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
import json
import gzip
import base64
import csv
from io import BytesIO
import scipy.io
from scipy.io import loadmat, savemat
from scipy.io import loadmat, savemat

from math import sqrt
import pingouin as pg

from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.linear_model import LinearRegression

from datetime import datetime, timedelta

import re
import os
import glob

from redcap import Project


#Stop pandas from downcasting during replace operations to silence the warning messages
pd.set_option('future.no_silent_downcasting', True)

########################################################
########################################################
########################################################
### Read in data

baseline_api = "1D481003114ECDA8E4077078E0D08D0A"
rpt_api = "66785AFE5341D3F73B4F518339C60186"
redcap_api = "https://redcap.research.yale.edu/api/"

# Load the REDCap project using API call
project = Project(redcap_api, rpt_api)

    
# Get the latest data
df = project.export_records(format_type='df')
df = df.reset_index(names='record_id')

#convert record id to integer
df['record_id']=df['record_id'].astype(int)

#Also get the decoder that tells us what timepoint each task version is done
df_decode = pd.read_excel(f'/Users/msg74/Desktop/powers_lab/Analysis/Aim1_rpt/data/resources/RptRandomizationDecoder_transformed_intNotString.xlsx')

#And convert the timepoints to strings
df_decode[[2,3,4,5]] = df_decode[[2,3,4,5]].replace({2:'hyp',3:'acu',4:'sub',5:'pers'})


######## Clean the data
# remove the test records
df = df[df['record_id']>67]

df_rpt_phone = df.copy()

########### SPECIFY RECORDS TO CHECK
### DEFAULT: Check all records that have finished at least hyperacute or acute but not had their QC completed. 

#Exclude people who have not finished one of the OG timepoints
newrecords = df = df[~(df['browser_hyp'].isna() & df['browser_acu'].isna())]

#Only check the records of people who have not been paid yet for each timepoint
newrecords_hyp = newrecords[newrecords['payment_date_hyp'].isna() & newrecords['browser_hyp'].notna()]['record_id'].tolist()
newrecords_acu = newrecords[newrecords['payment_date_acu'].isna() & newrecords['browser_acu'].notna()]['record_id'].tolist()
newrecords_sub = newrecords[newrecords['payment_date_sub'].isna() & newrecords['browser_sub'].notna()]['record_id'].tolist()
newrecords_pers = newrecords[newrecords['payment_date_pers'].isna() & newrecords['browser_pers'].notna()]['record_id'].tolist()

# newrecords['paid_rpt'] = newrecords['paid_rpt'].fillna(0)
# newrecords['paid_rpt'] = newrecords['paid_rpt'].astype(int)
# newrecords = newrecords[~(newrecords['paid_rpt']==1)]
# records_to_check = newrecords['record_id'].tolist()

df = df.reset_index()

# Go ahead and drop the fisher task
df = df.drop(columns=[x for x in df.columns if 'fisher' in x])

# Set EXCUSED tasks to nan so there isnt an error
df = df.replace({"EXCUSED": np.nan},regex=True)

#### Also load the latest baseline dataset
df_bl = Project(redcap_api, rpt_api).export_records(format_type='df')
df_bl = df_bl.reset_index(names='record_id')
testingrecords = ['BlueLightTesting_1','r/Drugs Testing','TESTING','TESTING2','TESTING3','TESTING4','TESTING5','TESTING6']
df_bl = df_bl[~(df_bl['record_id'].isin(testingrecords))]
df_bl['record_id']=df_bl['record_id'].astype(int)


#### Check whether or not they are using different phone numbers for the baseline and rpt

def comparevars(df1,df2,var1,var2):
    df1 = df1[df1[var1].notna()]
    df2 = df2[df2[var2].notna()]
    df1 = df1.reset_index()
    df2 = df2.reset_index()

    if var1 == var2:
        df2.rename(columns={var2:var2+'_bl'},inplace=True)
        var2 = var2+'_bl'
    
    compare_df = df1.merge(df2[['record_id',var2]], on='record_id', how='left')

    diff_df = compare_df[compare_df[var1]!=compare_df[var2]][["record_id",var1,var2]]

    if len(diff_df) > 0:
        print(f"Participants using different phone numbers between baseline and longitudinal: {diff_df}")

    return diff_df

wrongphones = comparevars(df_rpt_phone,df_bl,'phone_number','phone_number')



######### First, change the random task versions to be specific timepoints ###########

#change prl version name to 2,3,4,5 to match ach and vch
df = df.rename(columns={'task_data_prltask4':'task_data_prltask5',
'task_data_prltask3': 'task_data_prltask4',
'task_data_prltask2': 'task_data_prltask3',
'task_data_prltask1': 'task_data_prltask2'})
# ,
# 'task_data_fishertask4':'task_data_fishertask5',
# 'task_data_fishertask3': 'task_data_fishertask4',
# 'task_data_fishertask2': 'task_data_fishertask3',
# 'task_data_fishertask1': 'task_data_fishertask2'})


#Map version numbers to timepoints and assign values
df = df.reset_index()
for index, row in df.iterrows():
    random_rpt = row['random_rpt']
    for col in df.columns:
        if col.startswith('task_data') and col[-1].isdigit():  # Check if the column is a task_data column and ends with a digit
            version_number = int(col[-1])  # Get the version number from the column name
            task = col[10:(len(col)-1)] #Get the task name; since task_data_ is 10 characters, start with the 10th character cuz first number not inclusive and go to final character -1 to cut off the number
            timepoint = df_decode.loc[df_decode['random_rpt'] == random_rpt, version_number].values[0]  # Find the corresponding timepoint in dfdecode
            df.at[index,f'{task}_{timepoint}'] = row[col]



### If someone has retrieved data, use it instead
timepoints = ['hyp', 'acu', 'sub', 'pers']
# tasks = ['vch','prl','fisher','ach']
tasks = ['vch','prl','ach']

for index,row in df.iterrows():
    for timepoint in timepoints:
        for task in tasks:
            if task == 'vch':
                realtask = 'vch_short_psychedelic_'
            elif task == 'prl':
                realtask = 'prltask'
            elif task == 'ach':
                realtask = 'ach_task_short_psychedelic'
            elif task == 'fisher':
                realtask = 'fishertask'
            for i in range(2,6):
                if pd.isna(row[f'{realtask}_{timepoint}']):
                    if isinstance(row[f'task_data_{task}_retrieved_{i}_{timepoint}'],str):
                        df.at[index,f'{realtask}_{timepoint}'] = row[f'task_data_{task}_retrieved_{i}_{timepoint}']

#############################
### ACH QC ####
#############################

#### Create full trial-by-trial dataset and get useful values added to og dataset
def get_ch_trial_data(dataframe,task_data_taskname):
    participant_dfs = []
    participant_df = None  # Initialize participant_df to None


    for index, row in dataframe.iterrows():
        # Extract the JSON string from the column
        if isinstance(row[task_data_taskname],str):
            compressed_json = row.get(task_data_taskname, "")

            if compressed_json:

                #decompress and decode the JSON
                decoded_data = base64.b64decode(compressed_json)
                decompressed_json = gzip.decompress(decoded_data).decode('utf-8')

                #Parse JSON data
                data = json.loads(decompressed_json)

                #create dataframes of the variables we want for each block then concatenate into one for each participant, add to list of dataframes for later concatenation
                block_dfs = []
                blocks = ['component_1','component_2','component_3','component_4']
                block_num = 1
                for block in blocks:
                    df1 = pd.DataFrame()
                    df1['response'] = data[block]['response']
                    df1['rt'] = data[block]['responseTime']
                    df1['decibels'] = data[block]['decibels']
                    df1['component'] = block_num
                    df1['record_id']=row['record_id']

                    block_dfs.append(df1)
                    block_num+=1
                
                #add all the blocks into one dataframe
                participant_df = pd.concat(block_dfs,ignore_index=True)

                #create new column based on decibels for simple % of threshold
                participant_df['intensity'] = participant_df['decibels'].replace({data['processedData']['intensities']['c25']:25,
                data['processedData']['intensities']['c50']:50,
                data['processedData']['intensities']['c75']:75
                })

                #add each participant's dataframe into list of dataframes
                participant_dfs.append(participant_df)
                    
    # concatenate each participant's full CH data into one ENORMOUS dataframe
    if len(participant_dfs)>1:
        ch_data_all = pd.concat(participant_dfs,ignore_index=True)
    else:
        ch_data_all = participant_df
    return ch_data_all

### Function to, get slopes along with p-values into a dataframe for an entire ach_master dataframe
def test_detection_probability(data,intensity_var):
    results = []
    for record_id, group in data.groupby('record_id'):
        model = smf.logit(formula=f'response ~ {intensity_var}', data=group).fit(disp=0)
        p_value = model.pvalues[f'{intensity_var}']
        beta_coefficient = model.params[f'{intensity_var}']
        results.append({'record_id': record_id, 'p_value': p_value, 'beta_coefficient': beta_coefficient})
    
    results_df = pd.DataFrame(results)
    return results_df

### Function that calls the above functions to produce lists of people who fail ACH QC
def ch_qc(dataframe,taskname,get_trial_data,intensity_var):
    ach_master = get_trial_data(dataframe,taskname)
    ach_master['record_id']= ach_master['record_id'].astype('int64')
    ach_master['trial'] = ach_master.groupby('record_id').cumcount() + 1
    ### Check to make sure that none of the dataframes are identical using the reaction time column; if identical, someone copied and pasted the same string over and over.
    ach_master_pivoted = ach_master.pivot(index='record_id', columns='trial', values='rt')#record_id becomes the index
    # Find duplicated rows
    duplicated_rows = ach_master_pivoted[ach_master_pivoted.duplicated(keep=False)]
    # And get the record_ids for duplicated rows
    fraud_copy_paste_ach = duplicated_rows.index.tolist() 

    detection_slopes = test_detection_probability(ach_master[['record_id',intensity_var,'response']].dropna(),intensity_var)

    ### Second, test whether they have <50% detection of first 15 trials
    df_15 = ach_master[ach_master['trial']<16]
    detection_15 = df_15.groupby('record_id').mean().reset_index()

    #Get those who fail qc
    negative = detection_slopes[detection_slopes['beta_coefficient']<0]['record_id'].tolist()
    zero = detection_slopes[detection_slopes['p_value']>0.05]['record_id'].tolist()
    fail_first_fifteen = detection_15[detection_15['response']<0.5]['record_id'].tolist()

    return negative,zero,fail_first_fifteen,fraud_copy_paste_ach

############### Now run for each timepoint
#Hyperacute
negative_hyp, zero_hyp, fail_first_fifteen_hyp,fraud_copy_paste_ach_hyp = ch_qc(df,'ach_task_short_psychedelic_hyp',get_ch_trial_data,'decibels')
#Acute
negative_acu, zero_acu, fail_first_fifteen_acu,fraud_copy_paste_ach_acu = ch_qc(df,'ach_task_short_psychedelic_acu',get_ch_trial_data,'decibels')
#Subacute
negative_sub, zero_sub, fail_first_fifteen_sub,fraud_copy_paste_ach_sub = ch_qc(df,'ach_task_short_psychedelic_sub',get_ch_trial_data,'decibels')
#Persisting
try:
    negative_pers, zero_pers, fail_first_fifteen_pers,fraud_copy_paste_ach_pers = ch_qc(df,'ach_task_short_psychedelic_pers',get_ch_trial_data,'decibels')
except Exception as e:
    print(f"Can't do persisting timepoint because {e}")



# Manually pass record 1570 for ACH subacute
if 1570 in negative_sub:
    negative_sub.remove(1570)
if 1570 in zero_sub:
    zero_sub.remove(1570)
if 1570 in fail_first_fifteen_sub:
    fail_first_fifteen_sub.remove(1570)
if 1570 in fraud_copy_paste_ach_sub:
    fraud_copy_paste_ach_sub.remove(1570)
#############################
### VCH QC ####
#############################
#### Create full trial-by-trial dataset and get useful values added to og dataset
def get_vch_trial_data(dataframe,task_data_taskname):
    participant_dfs = []

    for index, row in dataframe.iterrows():
        # Extract the JSON string from the column
        if isinstance(row[task_data_taskname],str):
            compressed_json = row.get(task_data_taskname, "")

            #decompress and decode the JSON
            decoded_data = base64.b64decode(compressed_json)
            decompressed_json = gzip.decompress(decoded_data).decode('utf-8')

            #Parse JSON data
            data = json.loads(decompressed_json)

            #create dataframes of the variables we want for each block then concatenate into one for each participant, add to list of dataframes for later concatenation
            block_dfs = []
            blocks = ['component_1','component_2','component_3','component_4']
            block_num = 1
            for block in blocks:
                df1 = pd.DataFrame()
                df1['response'] = data[block]['response']
                df1['rt'] = data[block]['responseTime']
                df1['contrasts'] = data[block]['contrasts']
                df1['component'] = block_num
                df1['record_id']=data['recordId']

                block_dfs.append(df1)
                block_num+=1
            
            #add all the blocks into one dataframe
            participant_df = pd.concat(block_dfs,ignore_index=True)

            ## create new column based on decibels for simple % of threshold

            #Get the contrasts, round them to 5 digits (so replace doesn't freak out), get the contrasts sorted from smallest to largest, and then assign them as just 0,25,50,75
            participant_df['contrasts'] = participant_df['contrasts'].round(5)
            unique_values = participant_df['contrasts'].unique()
            unique_values_list = unique_values.tolist()
            unique_values_list.sort()

            participant_df['intensity'] = participant_df['contrasts'].replace({unique_values_list[1]:25,
            unique_values_list[2]:50,
            unique_values_list[3]:75
            })

            #add each participant's dataframe into list of dataframes
            participant_dfs.append(participant_df)

    # concatenate each participant's full CH data into one ENORMOUS dataframe
    if len(participant_dfs)>1:
        ch_data_all = pd.concat(participant_dfs,ignore_index=True)
    else:
        ch_data_all = participant_df
    return ch_data_all

############### VCH qc for each timepoint
#Hyperacute
vch_negative_hyp, vch_zero_hyp, vch_fail_first_fifteen_hyp,fraud_copy_paste_vch_hyp = ch_qc(df,'vch_short_psychedelic__hyp',get_vch_trial_data,'contrasts')
#Acute
vch_negative_acu, vch_zero_acu, vch_fail_first_fifteen_acu,fraud_copy_paste_vch_acu = ch_qc(df,'vch_short_psychedelic__acu',get_vch_trial_data,'contrasts')
#Subacute
vch_negative_sub, vch_zero_sub, vch_fail_first_fifteen_sub,fraud_copy_paste_vch_sub = ch_qc(df,'vch_short_psychedelic__sub',get_vch_trial_data,'contrasts')
#Persisting
vch_negative_pers, vch_zero_pers, vch_fail_first_fifteen_pers,fraud_copy_paste_vch_pers = ch_qc(df,'vch_short_psychedelic__pers',get_vch_trial_data,'contrasts')

# try:
#     vch_negative_pers, vch_zero_pers, vch_fail_first_fifteen_pers,fraud_copy_paste_vch_pers = ch_qc(df,'vch_short_psychedelic__pers',get_vch_trial_data,'contrasts')
# except Exception as e:
#     print(f"Can't do persisting timepoint because {e}")


#############################
### PRL QC ####
#############################
# Function that converts PRL JSON string into dataframe
def process_json_string(compressed_str):
    try:
        decoded_bytes = base64.b64decode(compressed_str)
        decompressed_str = gzip.decompress(decoded_bytes).decode('utf-8')
        data_dict = json.loads(decompressed_str)
        
        # Extract 'data' key, record id, and project id
        data_array = data_dict.get("data", [])
        record_id = data_dict.get('recordId', 'N/A')
        project_id = data_dict.get('projectId', 'N/A')
        
        # Add 'record_id' and 'projectId' to each row in data_array
        for row in data_array:
            row['record_id'] = record_id
            row['projectId'] = project_id
        
        return data_array
    except Exception as e:
        print(f"Error processing JSON string: {e}")
        return []

#####This function takes the number of trials that meet a certain criteria and then appends a column to the dataframe w/%of total trials meeting those criteria
def get_percent_trials_per_participant(dataframe,colname,colvaluetocount,newcolname):
    number_trials = dataframe[dataframe[colname]==colvaluetocount].groupby('record_id').size()
    total_trials = dataframe.groupby('record_id').size()
    percent_trials = (number_trials/total_trials) * 100
    percent_trials_df = percent_trials.reset_index(name=newcolname)
    newdataframe = dataframe.merge(percent_trials_df,on='record_id')
    return newdataframe

######## Comprehensive QC function that returns lists of records failing diff types of QC!
def prl_qc(dataframe,taskname):
    ### Convert all of the JSON strings into dataframes and then concatenate together. 
    # List to hold DataFrames
    dataframes_prl_list = []
    # For each subject, decompress the JSON string, convert to dataframe, add to list of dataframes
    for row in range(len(dataframe)):
        if isinstance(df.loc[row,taskname],str):
            compressed_string_prl = df.loc[row,taskname]
            decompressed_prl_data_array = process_json_string(compressed_string_prl)
            #convert to PANDAS dataframe
            decompressed_prl_df = pd.DataFrame(decompressed_prl_data_array)
            #append dataframe to list of dataframes
            dataframes_prl_list.append(decompressed_prl_df)

    #Concatenate dataframes
    if len(dataframes_prl_list)>0:
        df_prl = pd.concat(dataframes_prl_list, ignore_index=True)
    else:
        df_prl = dataframes_prl_list[0]
        
    #replace fractal1 etc. with just 1,2, and 3
    df_prl = df_prl.replace('fractal1', 1)
    df_prl = df_prl.replace('fractal2', 2)
    df_prl = df_prl.replace('fractal3', 3)

    #convert the fractal numbers to integers
    df_prl['choice'] = df_prl['choice'].astype(int)
    # at some point may want to replace -999 with NaN df_prl = df_prl.replace(-999,np.nan)

    #convert record_id to integer to match df
    df_prl['record_id'] = df_prl['record_id'].astype(int)

    df_prl.reset_index(drop=True,inplace=True)

    # #Replace -999 with NaNs
    # df_prl = df_prl.replace(-999,np.nan)

    ### Check to make sure that none of the dataframes are identical using the reaction time column; if identical, someone copied and pasted the same string over and over.
    prl_master_pivoted = df_prl.pivot(index='record_id', columns='trial', values='decisionTime')#record_id becomes the index
    # Find duplicated rows
    duplicated_rows_prl = prl_master_pivoted[prl_master_pivoted.duplicated(keep=False)]
    # Extract the record_ids of duplicated rows 
    fraud_copy_paste_prl = duplicated_rows_prl.index.tolist() 


    #Calculate the % of trials that each participant is picking the right answer and appends as column:
    df_prl = get_percent_trials_per_participant(dataframe=df_prl,colname='rewardProbChoice',colvaluetocount=0.85,newcolname='%Correct')

    ##Which participants are performing worse than chance? 
    grouped_record_id = df_prl.groupby('record_id')['%Correct'].mean().reset_index()
    worse_than_chance = []
    for row in range(len(grouped_record_id)):
        if grouped_record_id.loc[row,'%Correct']<34:
            worse_than_chance.append(grouped_record_id.loc[row,'record_id'])

    ##Which participants are not responding 10% of the time? 
    #Calculate the % of trials that each participant is not responding
    df_prl = get_percent_trials_per_participant(dataframe=df_prl,colname='keyChoice',colvaluetocount=-999,newcolname='%NoResponse')
    grouped_record_id = df_prl.groupby('record_id')['%NoResponse'].mean().reset_index()
    #now find who doesn't respond >10% of them time
    non_responders = []
    for row in range(len(grouped_record_id)):
        if grouped_record_id.iloc[row]['%NoResponse']>10:
            non_responders.append(grouped_record_id.loc[row,'record_id'])

    ##Determine whether each trial represents a win-stay or a lose shift
    win_stay_lose_stay = []
    for row in range(len(df_prl)):
        if (row-1) <0 or df_prl.loc[row,'record_id']!=df_prl.loc[row-1,'record_id'] or np.isnan(df_prl.loc[row,'outcome']) or df_prl.loc[(row-1),'outcome']==-999:
            win_stay_lose_stay.append(np.nan)    
        else:
            if df_prl.loc[(row-1),'outcome']==1 and df_prl.loc[row,'rewardProbChoice']==df_prl.loc[(row-1),'rewardProbChoice']:
                win_stay_lose_stay.append('win-stay')
            elif df_prl.loc[(row-1),'outcome']==1 and df_prl.loc[row,'rewardProbChoice']!=df_prl.loc[(row-1),'rewardProbChoice']:
                win_stay_lose_stay.append('win-switch')
            elif df_prl.loc[(row-1),'outcome']==0 and df_prl.loc[row,'rewardProbChoice']!=df_prl.loc[(row-1),'rewardProbChoice']:
                win_stay_lose_stay.append('lose-switch')
            elif df_prl.loc[(row-1),'outcome']==0 and df_prl.loc[row,'rewardProbChoice']==df_prl.loc[(row-1),'rewardProbChoice']:
                win_stay_lose_stay.append('lose-stay')
            else:
                print(df_prl.loc[row,'trial'])


    #Add this as a column            
    df_prl['win_stay_lose_switch'] = win_stay_lose_stay
    ## Get the percent win- and lose-stays
    df_prl = get_percent_trials_per_participant(dataframe=df_prl,colname='win_stay_lose_switch',colvaluetocount='win-stay',newcolname='win_stay_percent')
    df_prl = get_percent_trials_per_participant(df_prl,'win_stay_lose_switch','lose-stay','lose_stay_percent')


    ##Which participants do not exhibit win-stay behavior at least 33% of the time? 
    grouped_record_id = df_prl.groupby('record_id')['win_stay_percent'].mean().reset_index()
    low_win_stay = []
    for row in range(len(grouped_record_id)):
        if grouped_record_id.loc[row,'win_stay_percent']<34:
            low_win_stay.append(grouped_record_id.loc[row,'record_id'])

    ######## DISCUSS WITH AL AND ALEXANDRIA; DOES THIS MAKE SENSE TO KEEP? 
    ##Which participants do not exhibit ANY (less than 1% or <3 trials) lost-stay behavior, suggesting they did not understand the instructions?
    no_lose_stay = []
    grouped_record_id = df_prl.groupby('record_id')['lose_stay_percent'].mean().reset_index()
    for row in range(len(grouped_record_id)):
        if grouped_record_id.loc[row,'lose_stay_percent']<1:
            no_lose_stay.append(grouped_record_id.loc[row,'record_id'])

    return fraud_copy_paste_prl, no_lose_stay, worse_than_chance, non_responders

# Hyperacute
fraud_copy_paste_prl_hyp, no_lose_stay_hyp, worse_than_chance_hyp, non_responders_hyp = prl_qc(df, 'prltask_hyp')

# Acute
fraud_copy_paste_prl_acu, no_lose_stay_acu, worse_than_chance_acu, non_responders_acu = prl_qc(df, 'prltask_acu')

# Subacute
fraud_copy_paste_prl_sub, no_lose_stay_sub, worse_than_chance_sub, non_responders_sub = prl_qc(df, 'prltask_sub')

# Persisting
fraud_copy_paste_prl_pers, no_lose_stay_pers, worse_than_chance_pers, non_responders_pers = prl_qc(df, 'prltask_pers')

# try:
#     fraud_copy_paste_prl_pers, no_lose_stay_pers, worse_than_chance_pers, non_responders_pers = prl_qc(df, 'prltask_pers')
# except Exception as e:
#     print(f"Can't do persisting timepoint because {e}")

# #############################
# ### Fisher QC ####
# #############################

# def fisher_qc(dataframe,taskname):
#     ### Convert all of the JSON strings into dataframes and then concatenate together. 
#     # List to hold DataFrames
#     dataframes_fisher_list = []
#     # For each subject, decompress the JSON string, convert to dataframe, add to list of dataframes
#     for row in range(len(dataframe)):
#         if isinstance(df.loc[row,taskname],str):
#             compressed_string_fisher = df.loc[row,taskname]
#             decompressed_fisher_data_array = process_json_string(compressed_string_fisher)
#             #convert to PANDAS dataframe
#             decompressed_fisher_df = pd.DataFrame(decompressed_fisher_data_array)
#             #append dataframe to list of dataframes
#             dataframes_fisher_list.append(decompressed_fisher_df)

#     #Concatenate dataframes
#     if len(dataframes_fisher_list)>1:
#         df_fisher = pd.concat(dataframes_fisher_list,ignore_index=True)
#     else:
#         df_fisher = dataframes_fisher_list[0]

#     ### Basic data cleaning/prettifying
#     #convert record_id to integer to match df
#     df_fisher['record_id'] = df_fisher['record_id'].astype(int)

#     #Rename variables
#     df_fisher.replace(to_replace='stimuli/blank',value='blank',inplace=True)
#     df_fisher.replace(to_replace='stimuli/bead_g',value='green',inplace=True)
#     df_fisher.replace(to_replace='stimuli/bead_b',value='blue',inplace=True)
#     df_fisher.replace(to_replace='stimuli/bead_y',value='yellow',inplace=True)

#     #Remove correct or incorrect if participant did not hit the key for that trial
#     df_fisher['slow_or_fast']=np.nan
#     for index, row in df_fisher.iterrows():
#         if pd.isna(row['keyPress']):
#             df_fisher.at[index,'correct']=np.nan
#             if row['reactionTime']<1000:
#                 df_fisher.loc[index,'slow_or_fast']='fast'
#             else:
#                 df_fisher.loc[index,'slow_or_fast']='slow'

#     ### Check to make sure that none of the dataframes are identical using the reaction time column; if identical, someone copied and pasted the same string over and over.

#     #first need to drop the practice rows to prevent duplicates in pivot operation
#     df_fisher = df_fisher[df_fisher['session']=='real']

#     #and make new trials column so that trials build on each block
#     df_fisher['trials'] = df_fisher.groupby('record_id').cumcount() + 1

#     df_fisher.reset_index(drop=True,inplace=True)



#     #Pivot so that we get just a row of record_ids (where each column is a new trial) for each record
#     fisher_master_pivoted = df_fisher.pivot(index='record_id', columns='trials', values='reactionTime')#record_id becomes the index
#     # Find duplicated rows
#     duplicated_rows_fisher = fisher_master_pivoted[fisher_master_pivoted.duplicated(keep=False)]
#     # Extract the record_ids of duplicated rows 
#     fraud_copy_paste_fisher = duplicated_rows_fisher.index.tolist() 

#     ## QC: get people respond >10% of the time or guesssed correctly worse than chance
            
#     #Calculate % correct for each participant
#     df_fisher = get_percent_trials_per_participant(dataframe=df_fisher,colname='correct',colvaluetocount=True,newcolname='%Correct')

#     #Print who gets <33% correct
#     grouped_record_id = df_fisher.groupby('record_id')['%Correct'].mean().reset_index()
#     worse_than_chance_fisher = []
#     for row in range(len(grouped_record_id)):
#         if grouped_record_id.loc[row,'%Correct']<34:
#             worse_than_chance_fisher.append(grouped_record_id.loc[row,'record_id'])

#     #Calculate the % of trials that each participant is not responding
#     df_fisher = get_percent_trials_per_participant(dataframe=df_fisher,colname='keyPress',colvaluetocount=np.nan,newcolname='%NoResponse')
#     grouped_record_id = df_fisher.groupby('record_id')['%NoResponse'].mean().reset_index()
#     #now find who doesn't respond >10% of them time
#     non_responders_fisher = []
#     for row in range(len(grouped_record_id)):
#         if grouped_record_id.loc[row,'%NoResponse']>10:
#             non_responders_fisher.append(grouped_record_id.loc[row,'record_id'])

#     return fraud_copy_paste_fisher,worse_than_chance_fisher, non_responders_fisher



# ### And run the QC

# # Hyperacute
# fraud_copy_paste_fisher_hyp, worse_than_chance_fisher_hyp, non_responders_fisher_hyp = fisher_qc(df, 'fishertask_hyp')
# # Acute
# fraud_copy_paste_fisher_acu, worse_than_chance_fisher_acu, non_responders_fisher_acu = fisher_qc(df, 'fishertask_acu')
# # Subacute
# fraud_copy_paste_fisher_sub, worse_than_chance_fisher_sub, non_responders_fisher_sub = fisher_qc(df, 'fishertask_sub')
# # Persisting
# fraud_copy_paste_fisher_pers, worse_than_chance_fisher_pers, non_responders_fisher_pers = fisher_qc(df, 'fishertask_pers')

###############################

##############################
#### Check if they fail QC for each time point!
passed_qc = []

### Create payment sheet by adding a row for every time point for every participant
hyp_checks = negative_hyp + vch_negative_hyp + non_responders_hyp + worse_than_chance_hyp # + non_responders_fisher_hyp + worse_than_chance_fisher_hyp
acu_checks = negative_acu + vch_negative_acu + non_responders_acu + worse_than_chance_acu #+ non_responders_fisher_acu + worse_than_chance_fisher_acu
sub_checks = negative_sub + vch_negative_sub + non_responders_sub + worse_than_chance_sub #+ non_responders_fisher_sub + worse_than_chance_fisher_sub
pers_checks = negative_pers + vch_negative_pers + non_responders_pers + worse_than_chance_pers #+ non_responders_fisher_pers + worse_than_chance_fisher_pers

# Remove 1307 from hyp_checks
hyp_checks = [record for record in hyp_checks if record != 1307]

for record in newrecords_hyp:
    if pd.notna(df.loc[df['record_id']==record,'browser_hyp'].values[0]):
        print(f"\n {record} {df.loc[df['record_id']==record,'email_rpt'].values[0]} Finished Hyperacute:")
        if record in zero_hyp:
            print(f'{record} failed CH task (Hyperacute) due to slope not significantly different from zero')
        if record in negative_hyp:
            print(f'{record} failed CH task (Hyperacute) due to negative slope -- WE NEED TO CONTACT AL BECAUSE THEY MIGHT BE RANDOMLY RESPONDING')
        if record in fail_first_fifteen_hyp:
            print(f'{record} failed CH task (Hyperacute) due to too few responses for the first 15 trials -- not a big enough deal to care about')
        if record in vch_zero_hyp:
            print(f'{record} failed VCH task (Hyperacute) due to slope not significantly different from zero')
        if record in vch_negative_hyp:
            print(f'{record} failed VCH task (Hyperacute) due to negative slope -- WE NEED TO CONTACT AL BECAUSE THEY MIGHT BE RANDOMLY RESPONDING')
        if record in vch_fail_first_fifteen_hyp:
            print(f'{record} failed VCH task (Hyperacute) due to too few responses for the first 15 trials -- not a big enough deal to care about')
        if record in no_lose_stay_hyp:
            print(f'{record} failed PRL task (Hyperacute) due to not staying after a loss at least 3 times')
        if record in non_responders_hyp:
            print(f'{record} failed PRL task (Hyperacute) due to not responding >10% of the time')
        if record in worse_than_chance_hyp:
            print(f'{record} failed PRL task (Hyperacute) due to <33% correct responses')
        # if record in non_responders_fisher_hyp:
        #     print(f'{record} failed Fisher Task (Hyperacute) due to not responding >10% of the time')
        # if record in worse_than_chance_fisher_hyp:
        #     print(f'{record} failed Fisher Task (Hyperacute) due to <33% correct responses')
        if record in fraud_copy_paste_ach_hyp:
            print(f'{record} has identical ach RTs as someone else (Hyperacute)')
        if record in fraud_copy_paste_vch_hyp:
            print(f'{record} has identical vch RTs as someone else (Hyperacute)')
        if record in fraud_copy_paste_prl_hyp:
            print(f'{record} has identical prl RTs as someone else (Hyperacute)')
        # if record in fraud_copy_paste_fisher_hyp:
        #     print(f'{record} has identical fisher RTs as someone else (Hyperacute)') 
    if pd.isna(df.loc[df['record_id']==record,'browser_acu'].values[0]):
        print(f'\n{record} {df.loc[df['record_id']==record,'email_rpt'].values[0]} YET to complete the 24hr time point\n')

for record in newrecords_acu:
    if pd.notna(df.loc[df['record_id']==record,'browser_acu'].values[0]):
        print(f"\n {record} {df.loc[df['record_id']==record,'email_rpt'].values[0]} Finished acute:")
        if record in zero_acu:
            print(f'{record} failed CH task (acute) due to slope not significantly different from zero')
        if record in negative_acu:
            print(f'{record} failed CH task (acute) due to negative slope -- WE NEED TO CONTACT AL BECAUSE THEY MIGHT BE RANDOMLY RESPONDING')
        if record in fail_first_fifteen_acu:
            print(f'{record} failed CH task (acute) due to too few responses for the first 15 trials -- not a big enough deal to care about')
        if record in vch_zero_acu:
            print(f'{record} failed VCH task (acute) due to slope not significantly different from zero')
        if record in vch_negative_acu:
            print(f'{record} failed VCH task (acute) due to negative slope -- WE NEED TO CONTACT AL BECAUSE THEY MIGHT BE RANDOMLY RESPONDING')
        if record in vch_fail_first_fifteen_acu:
            print(f'{record} failed VCH task (acute) due to too few responses for the first 15 trials -- not a big enough deal to care about')
        if record in no_lose_stay_acu:
            print(f'{record} failed PRL task (acute) due to not staying after a loss at least 3 times')
        if record in non_responders_acu:
            print(f'{record} failed PRL task (acute) due to not responding >10% of the time')
        if record in worse_than_chance_acu:
            print(f'{record} failed PRL task (acute) due to <33% correct responses')
        # if record in non_responders_fisher_acu:
        #     print(f'{record} failed Fisher Task (acute) due to not responding >10% of the time')
        # if record in worse_than_chance_fisher_acu:
        #     print(f'{record} failed Fisher Task (acute) due to <33% correct responses')
        if record in fraud_copy_paste_ach_acu:
            print(f'{record} has identical ach RTs as someone else (acute)')
        if record in fraud_copy_paste_vch_acu:
            print(f'{record} has identical vch RTs as someone else (acute)')
        if record in fraud_copy_paste_prl_acu:
            print(f'{record} has identical prl RTs as someone else (acute)')
        # if record in fraud_copy_paste_fisher_acu:
        #     print(f'{record} has identical fisher RTs as someone else (acute)')
        if df.loc[df['record_id']==record,'less_sixhr_sincehyp_acu'].values[0]>0:
            print(f'{record} did the acute timepoint less than 6 hours after finishing the hyperacute timepoint')
        if df.loc[df['record_id']==record,'over_1day_sincehyp_acu'].values[0]>0:
            print(f'{record} did the acute timepoint more than 24 hours after finishing the hyperacute timepoint')

for record in newrecords_sub:
    if pd.isna(df.loc[df['record_id']==record,'browser_sub'].values[0]):
        print(f'\n{record} {df.loc[df['record_id']==record,'email_rpt'].values[0]} {record} {df.loc[df['record_id']==record,'email_rpt'].values[0]} YET to completed Afterglow time point\n')
    elif pd.notna(df.loc[df['record_id']==record,'browser_sub'].values[0]):
        print(f"\n {record} {df.loc[df['record_id']==record,'email_rpt'].values[0]} Finished subacute:")
        if record in zero_sub:
            print(f'{record} failed CH task (subacute) due to slope not significantly different from zero')
        if record in negative_sub:
            print(f'{record} failed CH task (subacute) due to negative slope -- WE NEED TO CONTACT AL BECAUSE THEY MIGHT BE RANDOMLY RESPONDING')
        if record in fail_first_fifteen_sub:
            print(f'{record} failed CH task (subacute) due to too few responses for the first 15 trials -- not a big enough deal to care about')
        if record in vch_zero_sub:
            print(f'{record} failed VCH task (subacute) due to slope not significantly different from zero')
        if record in vch_negative_sub:
            print(f'{record} failed VCH task (subacute) due to negative slope -- WE NEED TO CONTACT AL BECAUSE THEY MIGHT BE RANDOMLY RESPONDING')
        if record in vch_fail_first_fifteen_sub:
            print(f'{record} failed VCH task (subacute) due to too few responses for the first 15 trials -- not a big enough deal to care about')
        if record in no_lose_stay_sub:
            print(f'{record} failed PRL task (subacute) due to not staying after a loss at least 3 times')
        if record in non_responders_sub:
            print(f'{record} failed PRL task (subacute) due to not responding >10% of the time')
        if record in worse_than_chance_sub:
            print(f'{record} failed PRL task (subacute) due to <33% correct responses')
        # if record in non_responders_fisher_sub:
        #     print(f'{record} failed Fisher Task (subacute) due to not responding >10% of the time')
        # if record in worse_than_chance_fisher_sub:
        #     print(f'{record} failed Fisher Task (subacute) due to <33% correct responses')
        if record in fraud_copy_paste_ach_sub:
            print(f'{record} has identical ach RTs as someone else (subacute)')
        if record in fraud_copy_paste_vch_sub:
            print(f'{record} has identical vch RTs as someone else (subacute)')
        if record in fraud_copy_paste_prl_sub:
            print(f'{record} has identical prl RTs as someone else (subacute)')
        # if record in fraud_copy_paste_fisher_sub:
        #     print(f'{record} has identical fisher RTs as someone else (subacute)')
        if df.loc[df['record_id']==record,'days_sincehyp_sub'].values[0]>14:
            print(f'{record} did the subacute timepoint more than 14 days after hyperacute -- {df.loc[df['record_id']==record,'days_sincehyp_sub']} days')
        if df.loc[df['record_id']==record,'days_since_acu_sub'].values[0]>13:
            print(f'{record} did the subacute timepoint more than 13 days after acute -- {df.loc[df['record_id']==record,'days_since_acu_sub']} days')

for record in newrecords_pers:   
    if pd.isna(df.loc[df['record_id']==record,'browser_pers'].values[0]):
        print(f'\n{record} {df.loc[df['record_id']==record,'email_rpt'].values[0]} YET to completed Persisting time point\n')
    elif pd.notna(df.loc[df['record_id']==record,'browser_pers'].values[0]):
        print(f"\n {record} {df.loc[df['record_id']==record,'email_rpt'].values[0]} Finished persisting:")
        if record in zero_pers:
            print(f'{record} failed CH task (persisting) due to slope not significantly different from zero')
        if record in negative_pers:
            print(f'{record} failed CH task (persisting) due to negative slope -- WE NEED TO CONTACT AL BECAUSE THEY MIGHT BE RANDOMLY RESPONDING')
        if record in fail_first_fifteen_pers:
            print(f'{record} failed CH task (persisting) due to too few responses for the first 15 trials -- not a big enough deal to care about')
        if record in vch_zero_pers:
            print(f'{record} failed VCH task (persisting) due to slope not significantly different from zero')
        if record in vch_negative_pers:
            print(f'{record} failed VCH task (persisting) due to negative slope -- WE NEED TO CONTACT AL BECAUSE THEY MIGHT BE RANDOMLY RESPONDING')
        if record in vch_fail_first_fifteen_pers:
            print(f'{record} failed VCH task (persisting) due to too few responses for the first 15 trials -- not a big enough deal to care about')
        if record in no_lose_stay_pers:
            print(f'{record} failed PRL task (persisting) due to not staying after a loss at least 3 times')
        if record in non_responders_pers:
            print(f'{record} failed PRL task (persisting) due to not responding >10% of the time')
        if record in worse_than_chance_pers:
            print(f'{record} failed PRL task (persisting) due to <33% correct responses')
        # if record in non_responders_fisher_pers:
        #     print(f'{record} failed Fisher Task (persisting) due to not responding >10% of the time')
        # if record in worse_than_chance_fisher_pers:
        #     print(f'{record} failed Fisher Task (persisting) due to <33% correct responses')
        if record in fraud_copy_paste_ach_pers:
            print(f'{record} has identical ach RTs as someone else (persisting)')
        if record in fraud_copy_paste_vch_pers:
            print(f'{record} has identical vch RTs as someone else (persisting)')
        if record in fraud_copy_paste_prl_pers:
            print(f'{record} has identical prl RTs as someone else (persisting)')
        # if record in fraud_copy_paste_fisher_pers:
        #     print(f'{record} has identical fisher RTs as someone else (persisting)')
        if df.loc[df['record_id']==record,'days_sincehyp_pers'].values[0]>180:
            print(f'{record} did the subacute timepoint more than 6 months after hyperacute -- {df.loc[df['record_id']==record,'days_sincehyp_pers']} days')
        if df.loc[df['record_id']==record,'days_since_acu_pers'].values[0]>179:
            print(f'{record} did the subacute timepoint more than 6 months after acute -- {df.loc[df['record_id']==record,'days_since_acu_pers']} days')


# Function that will make all timestamp strings into the correct format:
def extract_redcap_date(date_str):
    # Nab the date separate from seconds
    truestring = date_str.split(" ",1)[0]

    # Extract datetime object and convert to classic mm/dd/yyyy format
    fmt = "%Y-%m-%d"
    dt = datetime.strptime(truestring, fmt)
    return dt.strftime("%m/%d/%Y")


# Function to update records in REDCap safely
def UpdateRedcap(records,fields,data,proj,newval=None,overwrite=None,convert_to_int=False):
    # Create a new dataframe with the records and fields
    update_df = data[data['record_id'].isin(records)]

    if newval:
        for field in fields:
            update_df[field] = newval

    allfields = fields + ['record_id']

    # Limit the dataframe to only the fields we want to update (and record_id cuz we need it)
    update_df = update_df[allfields]

    if convert_to_int==True:
        for field in allfields:
            update_df[field] = update_df[field].astype('Int64')
    if overwrite:
        proj.import_records(to_import=update_df,import_format='df',overwrite=overwrite)
    else:
        proj.import_records(to_import=update_df,import_format='df')


#### Set everything up for payment processing and REDcap updates
# Read in digitalcard_template for folks who need an extra sheet for digital gift cards
digitalcard_template_path = "/Users/msg74/Desktop/powers_lab/Analysis/Aim1_rpt/data/resources/digitalcard_template.csv"
digitalcard_template = pd.read_csv(digitalcard_template_path)

updates_for_redcap = df.copy()
expense_rows = []
redcap_update_rows = []  # Collect rows for REDCap update
for records_to_check, qclist, timepoint, time in zip(
    [newrecords_hyp, newrecords_acu, newrecords_sub, newrecords_pers],
    [hyp_checks, acu_checks, sub_checks, pers_checks],
    ['hyp', 'acu', 'sub', 'pers'],
    [1, 2, 3, 4]
):
    for record in records_to_check:

        event = f"Completion of Aim 8; timepoint {time}"

        # Get the one row for this record
        recorddf = df[df['record_id'] == record]
        recorddf = recorddf.reset_index(drop=True)

        row = recorddf.iloc[0]

        if record not in qclist:
            pay_pref = int(recorddf["payment_pref_rpt"])
            if pay_pref < 2:
                paytype = "Amazon-delivered physical VISA card"
                paydeets = row["payment_address_rpt"]
                comments = np.nan
            elif pay_pref < 3:
                paytype = 'Amazon Gift Card'
                paydeets = row["payment_email_rpt"]
                comments = np.nan
            elif pay_pref < 4:
                paytype = 'Digital US Bank Prepaid Gift Card'
                paydeets = row["payment_email_rpt"]
                comments = f"{row['firstname_rpt']} {row['lastname_rpt']}"
            else:
                paytype = "physical VISA mailed by third party postal carrier (eg. USPS/UPS)"
                paydeets = row["payment_address_rpt"]
                comments = np.nan


            name = myname

            if row[f"browser_{timepoint}"]:
                # Add to expense sheet
                thisrow = pd.DataFrame({'Date of Participation:  (month/day/year)':[extract_redcap_date(row[f"timestamp_{timepoint}_post"])],
                    'Date of Payment:          (month/day/year)': [np.nan],#[datetime.now().strftime("%m/%d/%Y")],
                    'Name of Yale Researcher requesting payment:':[name],
                    'Name of Yale Researcher providing payment:':["Catalina Mourgues"],#["Silmilly Toribio"],#["Maximillian S. Greenwald"],
                    'Amount of Payment:      (in USD)':["$60"],
                    'Subject ID #:':[row['record_id']],
                    'Type of Participation or Assessment Point: **Use the same verbiage as referenced in the HIC Economic Considerations** List all activities separately, including parking, miles, food or incidentals.':[event],
                    'Type of Payment:      (Cash, Gift Card, etc.)':[paytype],
                    'Delivery Detail (email, address)':[paydeets],
                    'Payment Instrument Link':[row[f"payment_url_{timepoint}"]],
                    'Comments /Confirmation': [comments]})
                # Append to the expense sheet
                expense_rows.append(thisrow)

                # --- Add row to digitalcard_template for digital gift card ---
                if paytype == 'Digital US Bank Prepaid Gift Card':
                    new_card_row = {col: np.nan for col in digitalcard_template.columns}
                    new_card_row['Load Amount'] = '$60'
                    new_card_row['Last Name '] = df.loc[df['record_id'] == record, 'lastname_rpt'].values[0]
                    new_card_row['First Name '] = df.loc[df['record_id'] == record, 'firstname_rpt'].values[0]
                    new_card_row['Email Address'] = paydeets
                    new_card_row['Fulfill by Client Program'] = '1'
                    new_card_row['Card Type'] = '1'
                    digitalcard_template = pd.concat([digitalcard_template, pd.DataFrame([new_card_row])], ignore_index=True)

                # If we are adding it to the expense sheet we should modify it in an update df to send back to update redcap
                updates_for_redcap.loc[updates_for_redcap['record_id'] == record, f'payment_date_{timepoint}'] = datetime.now().strftime("%m/%d/%Y")
                updates_for_redcap.loc[updates_for_redcap['record_id'] == record, f'payment_type_{timepoint}'] = paytype
                updates_for_redcap.loc[updates_for_redcap['record_id'] == record, f'payment_amount_{timepoint}'] = "$60"
                for col in [f"qc_passed_{timepoint}", f"qc_instructions_{timepoint}",f"send_pay_confirm_{timepoint}"]:
                    updates_for_redcap.loc[updates_for_redcap['record_id'] == record, col] = int(1)

        else:
            print(f"{record} ({row['email_rpt']}) failed QC for {timepoint} timepoint so leaving out of payment sheet for now...")
    

# Concatenate all the rows into an expense sheet -- And export to the csv file! 
if len(expense_rows) > 0: 
    expense_sheet = pd.concat(expense_rows, ignore_index=True)
    expense_sheetpath = f'{expensesheetpath}/expense_sheet_rpt_{datetime.now().strftime("%Y-%m-%d")}.csv'
    expense_sheet.to_csv(expense_sheetpath, index=False)
    if len(digitalcard_template) > 0:
        digitalcard_template_path = f'{expensesheetpath}/digitalcard_template_rpt_{datetime.now().strftime("%Y-%m-%d")}.csv'
        digitalcard_template.to_csv(digitalcard_template_path, index=False)
        print(f"Expense sheet saved to {expense_sheetpath} and Digital card template saved to {digitalcard_template_path}")
    else:
        print(f"Expense sheet saved to {expense_sheetpath}")
    print("check it out!")
    expense_sheet
else:
    print("Nobody has passed and requires payment.")

/var/folders/82/n_l5hkjx7k5gdb4371q2krc50_n_cp/T/ipykernel_1816/2474459893.py:159: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '...' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index,f'{task}_{timepoint}'] = row[col]
/var/folders/82/n_l5hkjx7k5gdb4371q2krc50_n_cp/T/ipykernel_1816/2474459893.py:159: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '...' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[index,f'{task}_{timepoint}'] = row[col]
/var/folders/82/n_l5hkjx7k5gdb4371q2krc50_n_cp/T/ipykernel_1816/2474459893.py:159: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'H4sIAAAAAAAAA929a49lR3Yd+F/qq1mB/X4Q8IeBPJY98Iwakkbw2BgYBFmWKLHJHrJagm3ovw/WKXbX


 2728 d4rkfr0st@gmail.com Finished subacute:
Expense sheet saved to /Users/msg74/Desktop/powers_lab/Analysis/Aim1_rpt/python_scripts/qc/expensesheets/expense_sheet_rpt_2026-03-13.csv
check it out!


/var/folders/82/n_l5hkjx7k5gdb4371q2krc50_n_cp/T/ipykernel_1816/2474459893.py:866: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  pay_pref = int(recorddf["payment_pref_rpt"])


# Run this cell once you have successfully [uploaded the payments to sharepoint](https://yaleedu-my.sharepoint.com/:x:/r/personal/silmilly_toribio_yale_edu/_layouts/15/Doc.aspx?sourcedoc=%7B989CA093-98FB-412F-A0BA-F99D26B72AAF%7D&file=Participant%20Payment%20Request%20Psychedelics%20MSG.xlsx&action=default&mobileredirect=true&DefaultItemOpen=1) and emailed Cata


In [2]:
#### Update REDCap 
### Select only rows that ended up in this expense sheet as payed
if expense_sheet.empty:
    print("No payments to update in REDCap. Move along, you silly goose.")
else:
    updates_for_redcap = updates_for_redcap[updates_for_redcap['record_id'].isin(expense_sheet['Subject ID #:'])]

    ### Select only record_id and columns related to payment for all timepoints as the df we will import to redcap
    import_cols = ['record_id']

    for timepoint in ['hyp','acu','sub','pers']:
        import_cols += [f'payment_date_{timepoint}',f'payment_amount_{timepoint}',f'qc_passed_{timepoint}',f'payment_type_{timepoint}',f"send_pay_confirm_{timepoint}"]
    
    #qc passed needs to be an integer:
    for col in import_cols:
        if ('qc_passed' in col) or ('send_pay_confirm' in col):
            updates_for_redcap[col] = updates_for_redcap[col].astype('Int64')

    updates_for_redcap_importdf = updates_for_redcap[import_cols]

    # Save a backup of the OG dataframe before updating REDCap
    df.to_csv(f"/Users/msg74/Desktop/powers_lab/Analysis/Aim1_rpt/data/backups/df_backup_{datetime.now().strftime('%Y-%m-%d')}.csv", index=False)
    print(f"Backup of df saved to /Users/msg74/Desktop/powers_lab/Analysis/Aim1_rpt/data/backups/df_backup_{datetime.now().strftime('%Y-%m-%d')}.csv")

    # Update REDCap with the payment information
    project.import_records(to_import=updates_for_redcap_importdf,import_format='df')
    print(f"Updated REDCap with payment information for {len(expense_sheet["Subject ID #:"].unique())} participants over {len(expense_sheet)} payment instruments")    

Backup of df saved to /Users/msg74/Desktop/powers_lab/Analysis/Aim1_rpt/data/backups/df_backup_2026-03-13.csv
Updated REDCap with payment information for 1 participants over 1 payment instruments


In [ ]:
# Update REDCap with the payment information
project.import_records(to_import=updates_for_redcap_importdf,import_format='df')
print(f"Updated REDCap with payment information for {len(expense_sheet["Subject ID #:"].unique())} participants over {len(expense_sheet)} payment instruments")    

In [ ]:
pomkin = df[df['record_id']==902]
print([x for x in pomkin.columns if 'prl' in x and 'complete' not in x and 'hyp' in x and 'retrieved' not in x and 'confirm' not in x])

In [ ]:
tasks = pomkin[[x for x in pomkin.columns if 'prl' in x and 'complete' not in x]]

In [ ]:
pomkin[['prltask_hyp','prltask_acu','prltask_sub','prltask_pers']]

In [ ]:
pomkin[[f"task_data_prltask{x}" for x in range(2,6)]]

In [ ]:
print(pomkin.at[5,'task_data_prltask3'])

In [ ]:
decode = df_decode.copy()
decode.columns = ['random_rpt',1,2,3,4]
decode

In [ ]:
### Add info here
downloads_path = "/Users/msg74/Downloads"
myname = "Maximillian S. Greenwald"

#Import libraries
import pandas as pd
import numpy as np
import math
import re
import statsmodels.api as sm
import statsmodels.formula.api as smf

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_regression
import json
import gzip
import base64
import csv
from io import BytesIO
import scipy.io
from scipy.io import loadmat, savemat
from scipy.io import loadmat, savemat

from math import sqrt
import pingouin as pg

from statsmodels.nonparametric.smoothers_lowess import lowess

from sklearn.linear_model import LinearRegression

from datetime import datetime, timedelta

import re
import os
import glob

from redcap import Project


#Stop pandas from downcasting during replace operations to silence the warning messages
pd.set_option('future.no_silent_downcasting', True)

########################################################
########################################################
########################################################
### Read in data

baseline_api = "1D481003114ECDA8E4077078E0D08D0A"
rpt_api = "66785AFE5341D3F73B4F518339C60186"
redcap_api = "https://redcap.research.yale.edu/api/"

# Load the REDCap project using API call
project = Project(redcap_api, rpt_api)

    
# Get the latest data
df = project.export_records(format_type='df')
df = df.reset_index(names='record_id')

#convert record id to integer
df['record_id']=df['record_id'].astype(int)
df = df[df['record_id']>67]

In [ ]:
df[['record_id','random_rpt']]

# Update REDCap 
### (only after you upload the payment csvs to:
https://yaleedu-my.sharepoint.com/:x:/r/personal/silmilly_toribio_yale_edu/_layouts/15/Doc.aspx?sourcedoc=%7B989CA093-98FB-412F-A0BA-F99D26B72AAF%7D&file=Participant%20Payment%20Request%20Psychedelics%20MSG.xlsx&action=default&mobileredirect=true&wdsle=0)

Update everyone in expense sheet:

In [ ]:
savemefornow = updates_for_redcap.copy()

In [ ]:
updates_for_redcap

In [ ]:
### Select only rows that ended up in this expense sheet as payed
updates_for_redcap = updates_for_redcap[updates_for_redcap['record_id'].isin(expense_sheet['Subject ID #:'])]

updates_for_redcap = updates_for_redcap[updates_for_redcap['record_id']<750]

### Select only record_id and columns related to payment for all timepoints as the df we will import to redcap
import_cols = ['record_id']
for timepoint in ['hyp','acu','sub','pers']:
    import_cols += [f'payment_date_{timepoint}',f'completion_date_{timepoint}',f'qc_passed_{timepoint}']#,f'qc_instructions_{timepoint}']

updates_for_redcap_importdf = updates_for_redcap[import_cols]

project.import_records(to_import=updates_for_redcap_importdf,import_format='df',date_format='MDY')

In [ ]:
project.import_records(to_import=updates_for_redcap_importdf,import_format='df',date_format='MDY')

Update select records:

In [ ]:
def UpdateRedcap(records,fields,data,proj,newval=None):
    
    # Create a new dataframe with the records and fields
    update_df = data[data['record_id'].isin(records)]

    if newval:
        for field in fields:
            update_df[field] = newval
    
    allfields = fields + ['record_id']
    update_df = update_df[allfields]
    proj.import_records(to_import=update_df,import_format='df')

In [ ]:
### STUFF FOR IMPORTING DATA INTO REDCAP
participantdf = df[df['record_id']==324]
participantdf
addme['record_id']='pycaptesting'

addmetrimmed = addme[['record_id',"questions_comments_ctrl"]]

testproj.import_records(to_import=addmetrimmed,import_format='df')

# Fiddling

## Get lists of invited participants and consented participants and who needs white noise added and who needs invites noted

In [ ]:
# Baseline Data
bl = pd.read_csv('/Users/msg74/Downloads/PsychedelicsAim1Onli_DATA_2024-12-17_1155.csv')
bl_emails = bl['interested_spstudy'].tolist()

#Get a df of records who we need to note have been invited to repeated measures
invited = pd.read_csv('/Users/msg74/Downloads/PsychedelicsAim1RepeatedMeasur_Participants_2024-12-17_1142.csv')
invited = invited[invited['Record']>65]
invite_emails = invited["Email Address"].tolist()
invited_recs = bl[bl['interested_spstudy'].isin(invite_emails)]['record_id'].tolist()

#People who signed consent (actually exist in main df)
consented = pd.read_csv('/Users/msg74/Downloads/PsychedelicsAim1Repe_DATA_2024-12-17_1144.csv')
consented = consented[~(consented['zoom_call_yn'].isna())]
consented_emails = consented['email_rpt'].tolist()
consented_recs = consented['record_id'].tolist()

#Get df for people who need white noise volume added
whitenoise = bl[bl['record_id'].isin(consented_recs)][['record_id','ach_vol_adj_amnt','ach_vol_adj_yn']]






In [ ]:
whitenoise

In [ ]:
invited_recs

In [ ]:
# # # Prompt user to confirm payments were uploaded to SharePoint
# confirm = input(f"Did you upload the payments at {expensesheetpath} to sharepoint? (yes/no): ").strip().lower()
# if confirm != "yes":
#     print("Please upload the payments to SharePoint before proceeding.")
# else:
#     today_str = datetime.now().strftime("%m/%d/%Y")
#     # For each row in expense_sheet, update REDCap fields
#     for idx, row in expense_sheet.iterrows():
#         record_id = row['Subject ID #:']
#         # Infer timepoint from 'Type of Participation or Assessment Point'
#         tp_str = str(row['Type of Participation or Assessment Point: **Use the same verbiage as referenced in the HIC Economic Considerations** List all activities separately, including parking, miles, food or incidentals.']).lower()
#         # Try to extract timepoint keyword
#         timepoint = None
#         for tp in ['hyp', 'acu', 'sub', 'pers']:
#             if tp in tp_str:
#                 timepoint = tp
#                 break
#         if timepoint is None:
#             print(f"Could not determine timepoint for record {record_id}, skipping.")
#             continue
#         # Prepare fields to update
#         fields = [f'payment_link_{timepoint}', f'qc_passed_{timepoint}', f'payment_date_{timepoint}']
#         # Set values: payment_link and qc_passed to 1, payment_date to today
#         update_df = updates_for_redcap[updates_for_redcap['record_id'] == record_id].copy()
#         update_df[f'payment_link_{timepoint}'] = 1
#         update_df[f'qc_passed_{timepoint}'] = 1
#         update_df[f'payment_date_{timepoint}'] = today_str
#         # Call UpdateRedcap with convert_to_int=True
#         UpdateRedcap([record_id], fields, updates_for_redcap, project, newval=None, overwrite=None, convert_to_int=True)
#     print("REDCap records updated for all records in the expense sheet.")